# Sign-off Comparison

Pulls the same four tabs out of three monthly sign-off workbooks and puts them side by
side: **TF 7599 DSP Rule** and the three **Experian DSP $75 Rule** bands.

The line names come from column **G, rows 9 to 35** - Avg Ending Receivables through
NET Inflow/Outflow - and the figures beside them from **H, I and J**, which are years 1,
2 and 3. That is the same layout the heat map reads: its unit C/O comes from H11/I11/J11,
which sits against `Cumulative Unit C/O Rate (%)` at G11.

Each line item carries its three sign-offs side by side, and `INCOME STATEMENT` and
`CASH FLOW` become the band across the top, since those rows name a section rather than
carry figures. One row per tab.

Out comes the same pair the heat map writes - `signoff_compare.html` to open or mail,
`signoff_compare_artifact.html` to publish, and `signoff_compare.xlsx` - minus the
actuals, which are not part of this.

Run the cells in order. The only cell you edit is the first one.

In [ ]:
# ── 1. Settings ───────────────────────────────────────────────────────────────
import datetime, math, os
import numpy as np
import pandas as pd
from openpyxl import Workbook, load_workbook
from openpyxl.styles import Alignment, Border, Font, PatternFill, Side
from openpyxl.utils import column_index_from_string, get_column_letter

PROJECT = "/Users/evernjoshua/Madam Claude Automation"
os.chdir(PROJECT)

# One workbook per sign-off, in the order they should read across the page.
# Edit the file names to match what is actually on disk.
BOOKS = [
    ("Sep-2026", "09'26 P&L Forecast - All.xlsm"),
    ("Oct-2026", "10'26 P&L Forecast - All.xlsm"),
    ("Nov-2026", "11'26 P&L Forecast - All.xlsm"),
]

# The tabs to pull out of every workbook, in the order they should stack.
SHEETS = [
    "TF 7599 DSP Rule",
    "Experian DSP $75 Rule <=600",
    "Experian DSP $75 Rule 601-639",
    "Experian DSP $75 Rule 640+",
]

# Column G carries the line names, rows 9 to 35, and H, I and J carry years 1,
# 2 and 3 beside them. That is the same layout the heat map reads: its unit C/O
# comes from H11/I11/J11, which sits against "Cumulative Unit C/O Rate (%)" at
# G11, and its ROA from H22/I22/J22 against "Cumulative ROA - Annualized" at G22.
LABEL_COLUMN = "G"
FIRST_ROW, LAST_ROW = 9, 35

# The figures run H through L and STOP there. Nothing to the right of L is read,
# whatever else the sheet holds - cell 3 drops any column named below that falls
# outside this range and says so.
FIRST_VALUE_COLUMN, LAST_VALUE_COLUMN = "H", "L"

# Which of those columns to show, and what to call each. Comment a line out and
# that period disappears from the table; the months repeat under each one that
# stays, period first.
YEARS = {
    "Yr 1": "H",
    "Yr 2": "I",
    "Yr 3": "J",
    "Yr 4": "K",
    "Yr 5": "L",
}

# A line with a name but no figures in any month is read as a section heading -
# INCOME STATEMENT, CASH FLOW - and groups the lines beneath it. Everything
# before the first heading sits under this name.
FIRST_SECTION = "Portfolio"

# These four were renamed: the $75 / 7599 names are current, the $49 / 4999 ones
# are what the same tabs were called before. An older sign-off still carries the
# old name, so each tab is looked up under its current name first and then under
# every former name here, newest first. The row keeps the current name whichever
# one matched, and cell 3 says which workbook used which - a rename partway
# through a series is exactly what quietly breaks a comparison.
SHEET_ALIASES = {
    "TF 7599 DSP Rule":              ["TF 4999 DSP Rule"],
    "Experian DSP $75 Rule <=600":   ["Experian DSP $49 Rule <=600"],
    "Experian DSP $75 Rule 601-639": ["Experian DSP $49 Rule 601-639"],
    "Experian DSP $75 Rule 640+":    ["Experian DSP $49 Rule 640+"],
}

# How each line is written. The first pattern that appears in the name wins;
# anything unmatched falls back to LAST. Edit freely - these are only display.
NUMBER_STYLES = [
    ("rate (%)",  dict(fmt="0.00%",         py="{:.2%}")),
    ("roa",       dict(fmt="0.00%",         py="{:.2%}")),
    ("(%)",       dict(fmt="0.00%",         py="{:.2%}")),
    ("accounts",  dict(fmt="#,##0",         py="{:,.0f}")),
    ("/accts",    dict(fmt='"$"#,##0.00',   py="${:,.2f}")),
    ("/avg",      dict(fmt='"$"#,##0.00',   py="${:,.2f}")),
    ("per ",      dict(fmt='"$"#,##0.00',   py="${:,.2f}")),
]
LAST = dict(fmt='"$"#,##0', py="${:,.0f}")

OUTPUT_XLSX = "signoff_compare.xlsx"
OUTPUT_HTML = "signoff_compare.html"

STAMP = datetime.datetime.now().strftime("%d %B %Y, %H:%M")

print(f"settings loaded - {len(BOOKS)} sign-offs, {len(SHEETS)} tabs, "
      f"names {LABEL_COLUMN}{FIRST_ROW}:{LABEL_COLUMN}{LAST_ROW}, "
      f"figures {FIRST_VALUE_COLUMN}-{LAST_VALUE_COLUMN}, "
      f"{len(YEARS)} period(s) shown")
for label, path in BOOKS:
    print(f"  {label:<9} {'FOUND    ' if os.path.exists(path) else 'NOT FOUND'}  {path}")

## 2. What the workbooks hold

Cell 3 prints the line items it found, grouped by section, and names any tab it could not
find along with the tabs that workbook does have. Copy a spelling out of that list into
`SHEETS` or `SHEET_ALIASES` if one does not match.

In [ ]:
# ── 3. Read the three workbooks ───────────────────────────────────────────────
NAN = float("nan")

def _squash(s):
    return "".join(str(s).split()).lower()

def _num(v):
    # cached Excel values arrive as numbers, strings, None or errors
    if v is None or isinstance(v, bool):
        return NAN
    if isinstance(v, (int, float)):
        return float(v)
    t = str(v).strip()
    if not t or t.startswith("#"):
        return NAN
    neg = t.startswith("(") and t.endswith(")")
    t = t.strip("()").replace(",", "").replace("$", "").replace(" ", "")
    pct = t.endswith("%")
    t = t.rstrip("%")
    try:
        x = float(t)
    except ValueError:
        return NAN
    if pct:
        x /= 100.0
    return -x if neg else x

def fmt_value(x, py):
    # the sign belongs outside the currency symbol: -$354,514, not $-354,514
    if x is None or (isinstance(x, float) and math.isnan(x)):
        return ""
    body = py.format(abs(x))
    return ("-" + body) if x < 0 else body

def style_for(label):
    low = str(label).lower()
    for needle, style in NUMBER_STYLES:
        if needle in low:
            return style
    return LAST

def names_for(name):
    # the current name first, then any former ones
    also = SHEET_ALIASES.get(name, [])
    if isinstance(also, str):
        also = [also]
    return [name] + list(also)

def find_sheet(wb, name):
    # (tab as the workbook spells it, the name it answered to) or (None, None)
    have = {_squash(a): a for a in wb.sheetnames}
    for candidate in names_for(name):
        hit = have.get(_squash(candidate))
        if hit:
            return hit, candidate
    return None, None

# Nothing outside H:L is ever read. A column named in YEARS that falls outside
# it is dropped here rather than quietly reaching into a neighbouring block.
_LO = column_index_from_string(FIRST_VALUE_COLUMN)
_HI = column_index_from_string(LAST_VALUE_COLUMN)
_OUT = {y: c for y, c in YEARS.items()
        if not (_LO <= column_index_from_string(c) <= _HI)}
if _OUT:
    print(f"outside {FIRST_VALUE_COLUMN}:{LAST_VALUE_COLUMN}, so not read: {_OUT}")
YEARS = {y: c for y, c in YEARS.items() if y not in _OUT}
if not YEARS:
    raise SystemExit(f"no value column left inside "
                     f"{FIRST_VALUE_COLUMN}:{LAST_VALUE_COLUMN}")

def read_book(path):
    # {tab: {row number: (label, {year: value})}} for one sign-off
    wb = load_workbook(path, data_only=True)
    try:
        out, missing = {}, []
        for name in SHEETS:
            actual, answered_to = find_sheet(wb, name)
            if actual is None:
                missing.append(name)
                continue
            if _squash(answered_to) != _squash(name):
                RENAMES.append((label_of(path), name, actual))
            ws = wb[actual]
            rows = {}
            for r in range(FIRST_ROW, LAST_ROW + 1):
                label = ws[f"{LABEL_COLUMN}{r}"].value
                label = " ".join(str(label).split()) if label is not None else ""
                rows[r] = (label, {y: _num(ws[f"{c}{r}"].value)
                                   for y, c in YEARS.items()})
            out[name] = rows
        if missing:
            print(f"    tabs NOT FOUND in {os.path.basename(path)}: {missing}")
            print(f"    tabs present: {wb.sheetnames}")
    finally:
        wb.close()
    return out

RENAMES = []            # [(month, current name, the name that tab actually has)]

def label_of(path):
    return next((m for m, p in BOOKS if p == path), os.path.basename(path))

BOOK_DATA, MONTHS = {}, []
for label, path in BOOKS:
    if not os.path.exists(path):
        print(f"skipping {label}: no file at {path!r}")
        continue
    BOOK_DATA[label] = read_book(path)
    MONTHS.append(label)
    print(f"{label}: {len(BOOK_DATA[label])} of {len(SHEETS)} tabs read from "
          f"{os.path.basename(path)}")

if not MONTHS:
    raise SystemExit("no workbooks could be read - check the file names in cell 1")

if RENAMES:
    print("\nmatched under a former name:")
    for month, current, actual in RENAMES:
        print(f"  {month}: {current!r} found as {actual!r}")

# ---- the line names, taken from the first month that has them ---------------
# A row with a name but no figures anywhere is a heading; a row with no name at
# all is a spacer and is dropped.
LINES = []            # [(row, section, label, style)]
section = FIRST_SECTION
for r in range(FIRST_ROW, LAST_ROW + 1):
    label = ""
    for m in MONTHS:
        for tab in BOOK_DATA[m].values():
            if tab.get(r) and tab[r][0]:
                label = tab[r][0]
                break
        if label:
            break
    if not label:
        continue                                   # spacer row
    has_value = any(not math.isnan(v)
                    for m in MONTHS
                    for tab in BOOK_DATA[m].values()
                    if tab.get(r)
                    for v in tab[r][1].values())
    if not has_value:
        section = label                            # INCOME STATEMENT, CASH FLOW
        continue
    LINES.append((r, section, label, style_for(label)))

print(f"\n{len(LINES)} line items across "
      f"{len(dict.fromkeys(s for _, s, _, _ in LINES))} section(s):")
shown = None
for r, sec, label, _ in LINES:
    if sec != shown:
        print(f"  [{sec}]")
        shown = sec
    print(f"    {LABEL_COLUMN}{r:<3} {label}")

In [ ]:
# ── 4. Build the table ────────────────────────────────────────────────────────
# One row per tab. Columns run section -> line item -> month, so each line item
# carries its three sign-offs side by side. With more than one year in YEARS the
# months repeat per year, year first.
SUBS = [(y, m) for y in YEARS for m in MONTHS]

def sub_label(year, month):
    return month if len(YEARS) == 1 else f"{year} {month}"

columns = [("Sheet", "", "")]
data = {("Sheet", "", ""): list(SHEETS)}

for r, section, label, _ in LINES:
    for year, month in SUBS:
        key = (section, label, sub_label(year, month))
        columns.append(key)
        data[key] = [
            BOOK_DATA.get(month, {}).get(tab, {}).get(r, ("", {}))[1].get(year, NAN)
            for tab in SHEETS
        ]

table = pd.DataFrame(data, columns=pd.MultiIndex.from_tuples(columns))
STYLE = {label: style for _, _, label, style in LINES}

# a tab that gave nothing back in any month is reported rather than left blank
empty = [tab for i, tab in enumerate(SHEETS)
         if all(math.isnan(v) for k, v in data.items()
                if k != ("Sheet", "", "") for v in [v[i]])]
if empty:
    print(f"no figures at all for: {empty}")

filled = sum(1 for k, col in data.items() if k != ("Sheet", "", "")
             for v in col if not math.isnan(v))
print(f"{len(SHEETS)} rows x {len(LINES)} line items x {len(SUBS)} column(s) "
      f"each - {filled} figures")
table.head()

In [ ]:
# ── 5. Preview it here ────────────────────────────────────────────────────────
from IPython.display import HTML, display

def preview(df):
    th = ("padding:4px 8px;border:1px solid #BFBFBF;background:#1F3864;color:#fff;"
          "font:600 11px/1.3 -apple-system,Segoe UI,sans-serif;text-align:center")
    th2 = th.replace("#1F3864", "#D9E1F2").replace("color:#fff", "color:#1F3864")
    th3 = th.replace("#1F3864", "#EEF2FA").replace("color:#fff", "color:#1F3864")
    td = ("padding:3px 8px;border:1px solid #E2E2E2;color:#1F2933;white-space:nowrap;"
          "font:11px/1.3 -apple-system,Segoe UI,sans-serif;text-align:right")
    n = len(SUBS)

    h = ["<div style='overflow:auto;max-height:640px'>"
         "<table style='border-collapse:collapse'>"]
    h.append("<tr><th rowspan=3 style='%s'>Sheet</th>" % th)
    for sec in dict.fromkeys(s for _, s, _, _ in LINES):
        span = n * sum(1 for _, s, _, _ in LINES if s == sec)
        h.append("<th colspan=%d style='%s'>%s</th>" % (span, th, sec))
    h.append("</tr><tr>")
    for _, _, label, _ in LINES:
        h.append("<th colspan=%d style='%s'>%s</th>" % (n, th2, label))
    h.append("</tr><tr>")
    for _, _, label, _ in LINES:
        for year, month in SUBS:
            h.append("<th style='%s'>%s</th>" % (th3, sub_label(year, month)))
    h.append("</tr>")

    for i, tab in enumerate(SHEETS):
        h.append("<tr><td style='%s;text-align:left;font-weight:600'>%s</td>" % (td, tab))
        for _, sec, label, style in LINES:
            for year, month in SUBS:
                x = df[(sec, label, sub_label(year, month))].iloc[i]
                h.append("<td style='%s'>%s</td>" % (td, fmt_value(x, style["py"])))
        h.append("</tr>")
    h.append("</table></div>")
    return "".join(h)

display(HTML(preview(table)))

In [ ]:
# ── 6. Write the workbook ─────────────────────────────────────────────────────
def _v(x):
    return None if x is None or (isinstance(x, float) and math.isnan(x)) else float(x)

def write_xlsx(df, path=None):
    path = OUTPUT_XLSX if path is None else path
    wb = Workbook()
    ws = wb.active
    ws.title = "Sign-offs"

    head  = PatternFill("solid", fgColor="1F3864")
    mid   = PatternFill("solid", fgColor="D9E1F2")
    low   = PatternFill("solid", fgColor="EEF2FA")
    hfont = Font(bold=True, color="FFFFFF", size=11)
    sfont = Font(bold=True, color="1F3864", size=10)
    thin  = Side(style="thin", color="BFBFBF")
    edge  = Border(left=thin, right=thin, top=thin, bottom=thin)
    mid_c = Alignment(horizontal="center", vertical="center", wrap_text=True)
    n = len(SUBS)

    c = ws.cell(1, 1, "Sheet")
    c.fill, c.font, c.alignment, c.border = head, hfont, mid_c, edge
    ws.merge_cells(start_row=1, start_column=1, end_row=3, end_column=1)
    for r in (2, 3):
        ws.cell(r, 1).fill, ws.cell(r, 1).border = head, edge

    col = 2
    for sec in dict.fromkeys(s for _, s, _, _ in LINES):
        span = n * sum(1 for _, s, _, _ in LINES if s == sec)
        t = ws.cell(1, col, sec)
        t.fill, t.font, t.alignment = head, hfont, mid_c
        ws.merge_cells(start_row=1, start_column=col, end_row=1, end_column=col + span - 1)
        for k in range(span):
            ws.cell(1, col + k).border = edge
        col += span

    col = 2
    for _, _, label, _ in LINES:
        t = ws.cell(2, col, label)
        t.fill, t.font, t.alignment = mid, sfont, mid_c
        ws.merge_cells(start_row=2, start_column=col, end_row=2, end_column=col + n - 1)
        for k in range(n):
            ws.cell(2, col + k).border = edge
            s = ws.cell(3, col + k, sub_label(*SUBS[k]))
            s.fill, s.font, s.alignment, s.border = low, sfont, mid_c, edge
        col += n
    last_col = col - 1

    for i, tab in enumerate(SHEETS):
        row = 4 + i
        t = ws.cell(row, 1, tab)
        t.border, t.font = edge, Font(bold=True, size=10)
        t.alignment = Alignment(vertical="center", wrap_text=True)
        col = 2
        for _, sec, label, style in LINES:
            for k, (year, month) in enumerate(SUBS):
                v = ws.cell(row, col + k,
                            _v(df[(sec, label, sub_label(year, month))].iloc[i]))
                v.number_format = style["fmt"]
                v.border = edge
                v.alignment = Alignment(horizontal="right")
            col += n

    ws.column_dimensions["A"].width = 32
    for k in range(2, last_col + 1):
        ws.column_dimensions[get_column_letter(k)].width = 14
    ws.row_dimensions[1].height = 20
    ws.row_dimensions[2].height = 34
    ws.row_dimensions[3].height = 16
    ws.freeze_panes = "B4"
    ws.sheet_view.showGridLines = False
    wb.save(path)
    return os.path.abspath(path)

print("written:", write_xlsx(table))

In [ ]:
# ── 7. Write the shareable page ───────────────────────────────────────────────
# Two files from the same content:
#   signoff_compare.html           - standalone, open it or mail it
#   signoff_compare_artifact.html  - same without the <html>/<body> wrapper,
#                                    which is what the Artifact publisher wants
from html import escape

_CSS = """
:root{--bg:#F7F7F5; --panel:#fff; --panel-2:#FAFAFC; --ink:#1F2933;
  --ink-2:#5A6472; --ink-3:#98A1AE; --rule:#E4E6EB; --rule-2:#D3D7DE;
  --accent:#1F3864; --head:#1F3864;}
@media (prefers-color-scheme:dark){:root:not([data-theme="light"]){
  --bg:#15171C; --panel:#1B1E25; --panel-2:#20242D; --ink:#E8EAEE;
  --ink-2:#AEB6C2; --ink-3:#79828F; --rule:#2A2F39; --rule-2:#39404C;
  --accent:#9BB4E8; --head:#223055;}}
:root[data-theme="dark"]{--bg:#15171C; --panel:#1B1E25; --panel-2:#20242D;
  --ink:#E8EAEE; --ink-2:#AEB6C2; --ink-3:#79828F; --rule:#2A2F39;
  --rule-2:#39404C; --accent:#9BB4E8; --head:#223055;}
*{box-sizing:border-box}
body{margin:0; background:var(--bg); color:var(--ink);
  font:14px/1.5 "IBM Plex Sans",-apple-system,Segoe UI,sans-serif;
  -webkit-font-smoothing:antialiased}
.wrap{max-width:1680px; margin:0 auto; padding:34px 16px 60px}
.mast{border-bottom:1px solid var(--rule); padding-bottom:18px; margin-bottom:22px}
.eyebrow{display:block; font:600 11px/1 "IBM Plex Mono",monospace;
  letter-spacing:.1em; text-transform:uppercase; color:var(--ink-3)}
h1{font:600 30px/1.2 Newsreader,Georgia,serif; margin:10px 0 0}
.panel{background:var(--panel); border:1px solid var(--rule); border-radius:12px;
  overflow:hidden}
.scroll{overflow:auto; max-height:78vh}
table{border-collapse:separate; border-spacing:0; width:max-content;
  min-width:100%; font-variant-numeric:tabular-nums}
th,td{white-space:nowrap}
th{position:sticky; background:var(--head); color:#fff; z-index:2;
  font:600 11px/1.25 "IBM Plex Sans",sans-serif; text-align:center;
  padding:6px 10px; border-bottom:1px solid var(--rule)}
th.sec{top:0; height:30px; letter-spacing:.04em; text-transform:uppercase}
th.item{top:30px; height:42px; background:var(--panel-2); color:var(--ink);
  font-weight:600; white-space:normal; min-width:96px;
  border-bottom:1px solid var(--rule-2)}
th.mon{top:72px; height:26px; background:var(--panel); color:var(--ink-2);
  font-weight:500; font-size:11px; border-bottom:1px solid var(--rule-2)}
th.corner{left:0; z-index:4; top:0; width:260px; text-align:left;
  vertical-align:bottom; padding-bottom:8px}
td{padding:0 10px; height:34px; font-size:12.5px; text-align:right;
  color:var(--ink); border-bottom:1px solid var(--rule)}
td.tab{position:sticky; left:0; z-index:1; background:var(--panel);
  text-align:left; font-weight:600; font-size:13px;
  border-right:1px solid var(--rule-2); white-space:normal; max-width:260px;
  padding:8px 14px; line-height:1.3}
td.first{border-left:2px solid var(--rule-2)}
th.item.first, th.mon.first{border-left:2px solid var(--rule-2)}
tbody tr:nth-child(even) td{background:var(--panel-2)}
tbody tr:nth-child(even) td.tab{background:var(--panel-2)}
td.blank{color:var(--ink-3)}
.sig{margin-top:18px; display:flex; flex-wrap:wrap; gap:16px;
  font:11px/1.6 "IBM Plex Mono",monospace; color:var(--ink-3)}
.sig code{font:inherit; color:var(--ink-2)}
@media (max-width:720px){.wrap{padding:22px 16px 40px} h1{font-size:23px}
  .scroll{max-height:70vh} th.corner, td.tab{width:170px; max-width:170px}}
"""

def _cell(x, py):
    txt = fmt_value(x, py)
    return (txt, "") if txt else ("-", "blank")

def build_page(df, fragment=False):
    sections = list(dict.fromkeys(s for _, s, _, _ in LINES))
    n = len(SUBS)

    h = ['<div class="panel"><div class="scroll"><table><thead>']
    h.append('<tr><th class="corner" rowspan="3" scope="col">Sheet</th>')
    for si, sec in enumerate(sections):
        span = n * sum(1 for _, s, _, _ in LINES if s == sec)
        h.append('<th class="sec" colspan="%d" scope="colgroup">%s</th>'
                 % (span, escape(sec)))
    h.append('</tr><tr>')
    for i, (_, _, label, _) in enumerate(LINES):
        h.append('<th class="item%s" colspan="%d" scope="colgroup">%s</th>'
                 % (" first" if i else "", n, escape(label)))
    h.append('</tr><tr>')
    for i, (_, _, label, _) in enumerate(LINES):
        for k, (year, month) in enumerate(SUBS):
            first = " first" if (i and k == 0) else ""
            h.append('<th class="mon%s" scope="col">%s</th>'
                     % (first, escape(sub_label(year, month))))
    h.append('</tr></thead><tbody>')

    for i, tab in enumerate(SHEETS):
        h.append('<tr><td class="tab">%s</td>' % escape(str(tab)))
        for j, (_, sec, label, style) in enumerate(LINES):
            for k, (year, month) in enumerate(SUBS):
                txt, cls = _cell(df[(sec, label, sub_label(year, month))].iloc[i],
                                 style["py"])
                first = " first" if (j and k == 0) else ""
                h.append('<td class="%s%s">%s</td>' % (cls, first, txt))
        h.append('</tr>')
    h.append('</tbody></table></div></div>')
    table_html = "".join(h)

    books = " ".join('<span>%s: <code>%s</code></span>'
                     % (escape(m), escape(os.path.basename(p)))
                     for m, p in BOOKS if m in MONTHS)
    if RENAMES:
        books += "".join('<span>%s read <code>%s</code> as <code>%s</code></span>'
                         % (escape(month), escape(actual), escape(current))
                         for month, current, actual in RENAMES)
    page = ('<header class="mast"><span class="eyebrow">%s &middot; %s</span>'
            '<h1>Sign-off Comparison</h1></header>\n%s\n'
            '<div class="sig">%s<span>Generated %s</span></div>'
            % (escape(" / ".join(MONTHS)), escape(STAMP), table_html, books,
               escape(STAMP)))

    head = ('<meta charset="utf-8">\n<title>Sign-off Comparison</title>\n'
            '<link rel="stylesheet" href="https://fonts.googleapis.com/css2?'
            'family=Newsreader:opsz,wght@6..72,400;6..72,600&'
            'family=IBM+Plex+Sans:wght@400;500;600&'
            'family=IBM+Plex+Mono:wght@400;500&display=swap">\n'
            '<style>%s</style>' % _CSS)
    body = '<main class="wrap">%s</main>' % page
    if fragment:
        return head + "\n" + body
    return ('<!doctype html>\n<html lang="en">\n<head>\n'
            '<meta name="viewport" content="width=device-width,initial-scale=1">\n'
            '%s\n</head>\n<body>\n%s\n</body>\n</html>\n' % (head, body))

def write_html(df, path=None):
    path = OUTPUT_HTML if path is None else path
    alt = os.path.splitext(path)[0] + "_artifact.html"
    with open(path, "w", encoding="utf-8") as fh:
        fh.write(build_page(df, fragment=False))
    with open(alt, "w", encoding="utf-8") as fh:
        fh.write(build_page(df, fragment=True))
    return os.path.abspath(path), os.path.abspath(alt)

page_path, frag_path = write_html(table)
print("open this one:    ", page_path)
print("publish this one: ", frag_path)